# 00 · Load and clean

Applies the cleaning rules in `src/metrics.py` (the same rules as the Power BI model's Power Query), saves
`data/flights.parquet`, and **asserts the totals match the Power BI report**.

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from src.metrics import load_clean, headline, DATA_DIR, OUTPUT_DIR, DATA_END
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
NAVY, VIOLET, LILAC, ORANGE, SLATE, GREEN = "#030734", "#682EFF", "#CD83FF", "#F05100", "#62748E", "#009588"
sns.set_theme(style="whitegrid", palette=[VIOLET, LILAC, NAVY, ORANGE, SLATE, GREEN])
FIG = OUTPUT_DIR / "figures"; FIG.mkdir(parents=True, exist_ok=True); DATA_DIR.mkdir(parents=True, exist_ok=True)
Cr = lambda v: f"₹{v/1e7:,.2f} Cr"; Lk = lambda v: f"₹{v/1e5:,.1f} L"
findings = []
def note(q, topic, text):
    findings.append(dict(q=q, topic=topic, finding=text)); print(f"[{q}] {topic}: {text}")
def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()
def kpis(g):
    return pd.Series({"flights": len(g), "revenue": g.Revenue.sum(), "pax": g.PaxCount.sum(), "seats": g.Capacity.sum(),
                      "ask": g.ask.sum(), "rpk": g.rpk.sum(), "lf": g.PaxCount.sum() / g.Capacity.sum(),
                      "fare": g.Revenue.sum() / g.PaxCount.sum(), "rask": g.Revenue.sum() / g.ask.sum(),
                      "yield_rpk": g.Revenue.sum() / g.rpk.sum(), "stage_km": g.ask.sum() / g.Capacity.sum()})

In [2]:
d = load_clean(); f = d["flights"]
h = headline(f); h

flights                             40,175.000
passengers                       2,214,543.000
seats                            3,767,214.000
revenue                        648,972,690.000
ask                          4,142,864,685.000
rpk                          2,424,978,029.000
load_factor                              0.588
rask                                     0.157
yield_per_rpk                            0.268
routes                                 179.000
distance_corrected_flights           5,660.000
fare_outliers                           86.000
dtype: float64

In [3]:
# Totals read from the Power BI model after refresh (pbi dax). Python must reproduce them exactly.
PBI = {"flights": 40175, "passengers": 2214543, "seats": 3767214, "revenue": 648972690,
       "routes": 179, "distance_corrected_flights": 5660, "fare_outliers": 86,
       "ask": 4142864685, "rpk": 2424978029}
rec = pd.DataFrame({"python": h[list(PBI)], "power_bi": pd.Series(PBI)}); rec["match"] = rec.python.round(0) == rec.power_bi
assert rec["match"].all(), rec
rec

,python,power_bi,match
flights,"40,175.000",40175,True
passengers,"2,214,543.000",2214543,True
seats,"3,767,214.000",3767214,True
revenue,"648,972,690.000",648972690,True
routes,179.000,179,True
distance_corrected_flights,"5,660.000",5660,True
fare_outliers,86.000,86,True
ask,"4,142,864,685.000",4142864685,True
rpk,"2,424,978,029.000",2424978029,True


In [4]:
f.to_parquet(DATA_DIR / "flights.parquet", index=False)
f.shape

(40175, 25)